In [1]:
import sys
from pathlib import Path

In [2]:
# in jupyter (lab / notebook), based on notebook path
module_path = str(Path.cwd().parents[0])

# # in standard python
# module_path = str(Path.cwd(__file__).parents[0] / "py")

if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import wandb

import data.donut_dataset as donut_dataset
import utils.helpers as helpers
import utils.confidential as confidential
import pytorch_lightning as pl

from datasets import load_dataset
from torch.utils.data import DataLoader
from data.donut_dataset import DonutDataset
from data.donut_pytorch_lightning_dataloader import DonutDataPLModuleCustom
from models.donut_pytorch_lightning import DonutModelPLModule
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import EarlyStopping

/Users/WilliamLiu/miniforge3/envs/HeR-T/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
image_path = "/Users/WilliamLiu/HeR_T_retaining/data/img_test"
max_length = 768

In [5]:
dataset = donut_dataset.data_loader(image_path)

print('Data Loading completes.')

this is the dataset DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 53
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 30
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 30
    })
})
Data Loading completes.


In [6]:
processor, model = donut_dataset.model_loader(dataset, max_length, "naver-clova-ix/donut-base")

print('Processor and Model are loaded.')

/Users/WilliamLiu/miniforge3/envs/HeR-T/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processor and Model are loaded.


In [7]:
processor.image_processor.size = helpers.image_size(dataset)
processor.image_processor.do_align_long_axis = False

print("Processor's setting is completed.")

Processor's setting is completed.


In [8]:
train_dataset = DonutDataset(image_path, max_length=max_length,
                             split="train", task_start_token="<s_herbarium>", 
                             prompt_end_token="<s_herbarium>",
                             sort_json_key=False, # cord dataset is preprocessed -> no need for this
                             model=model, 
                             processor=processor,
                             )

val_dataset = DonutDataset(image_path, max_length=max_length,
                             split="validation", task_start_token="<s_herbarium>", 
                             prompt_end_token="<s_herbarium>",
                             sort_json_key=False, # cord dataset is preprocessed -> no need for this
                             model=model,
                             processor=processor,
                             )

print("DonutDataset is loaded.")

/Users/WilliamLiu/miniforge3/envs/HeR-T/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:890: UserWarning: Truncated File Read
  warnings.warn(str(msg))


DonutDataset is loaded.


In [9]:
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s_herbarium>'])[0]

print("Model's setting is completed.")

Model's setting is completed.


In [23]:
config = {
    'max_epochs': 20,
    'val_check_interval': 0.25,
    'check_val_every_n_epoch': 1,
    'gradient_clip_val': 1.0,
    'num_training_samples_per_epoch': 36760,
    'lr': 2.5e-5, # or 2e-5
    'weight_decay': 2e-5,
    'dropout_rate': 0.2,
    'train_batch_sizes': [8],
    'val_batch_sizes': [1],
    'num_nodes': 1,
    'warmup_steps': 2500,
    'result_path': "/Users/WilliamLiu/HeR_T_retaining/results",
    'verbose': True, 
    'seed': 16
}

In [21]:
data_module = DonutDataPLModuleCustom(config=config)
data_module.train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2)
data_module.val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

print('Training and validation data loaders are created.')

Training and validation data loaders are created.


In [14]:
model_lightning = DonutModelPLModule(config, processor, model)

print('PyTorch Lightning Model has been set up.')

PyTorch Lightning Model has been set up.


In [15]:
# api key so that it doesn't ask me for it
wandb.login(key=confidential.api_key)
wandb_logger = WandbLogger(project="HeR-T-trial", name="localTrial")

# use default patiente
early_stop_callback = EarlyStopping(monitor="val_edit_distance", verbose=True, mode="min") 

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: weiweiliu (units). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/WilliamLiu/.netrc


In [16]:
class PushToHubCallback(pl.Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        print(f"Pushing model to the hub, epoch {trainer.current_epoch}")
        pl_module.processor.push_to_hub("Jac-Zac/thesis_donut",
                                    commit_message=f"Training in progress, epoch {trainer.current_epoch}")
        pl_module.model.push_to_hub("Jac-Zac/thesis_donut",
                                    commit_message=f"Training in progress, epoch {trainer.current_epoch}")
    def on_train_end(self, trainer, pl_module):
        print(f"Pushing model to the hub after training")
        pl_module.processor.push_to_hub("Jac-Zac/thesis_donut",
                                    commit_message=f"Training done")
        pl_module.model.push_to_hub("Jac-Zac/thesis_donut",
                                    commit_message=f"Training done")

In [19]:
trainer = pl.Trainer(
        accelerator="mps",
        #accelerator="gpu",
        devices=1,
        # strategy="xla_debug",
        max_epochs=config['max_epochs'],
        val_check_interval=config['val_check_interval'],
        check_val_every_n_epoch=config['check_val_every_n_epoch'],
        gradient_clip_val=config['gradient_clip_val'],
        precision="16-mixed", # we'll use mixed precision
        #precision=16, # we'll use mixed precision
        num_sanity_val_steps=0,
        logger=wandb_logger,
        callbacks=[PushToHubCallback(), early_stop_callback]
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [24]:
# trainer.fit(model_module, ckpt_path = '/kaggle/working/output/Donut/version_None/checkpoints/epoch=0-step=32164-v1.ckpt')
trainer.fit(model_lightning, datamodule=data_module)

AttributeError: 'DataLoader' object has no attribute '__code__'